# MCGA-LM: producing the run outputs on Kaggle

Runs the paper's experiments with the LLaMA-3-8B-Instruct configuration (`configs/llm_hf.yaml`), **one stage per Kaggle session**, carrying outputs forward between sessions.

## One-time setup
1. **Settings → Accelerator:** `GPU T4 x2`. **Settings → Internet:** on (needs a phone-verified Kaggle account).
2. **Add-ons → Secrets:** add `HF_TOKEN`, a Hugging Face token from an account that has accepted the licence at huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct.
3. **CLAS:** download it from IEEE DataPort (doi:10.21227/ybsw-yr53), upload the unzipped `CLAS_Database` folder as a **private** Kaggle dataset, and add it as input. WESAD and MAMEM are downloaded automatically.
4. Set `COMMIT` in the next cell to the pushed commit that contains the loader fixes, and keep it the same for every stage.

## Each session
1. Set `STAGE` in the next cell. Run the stages in this order:
   `pretrain` → `pilot` → `eval_MCGA-LM` → `eval_RAG-LLM` → `eval_M-LLM` → `eval_LLM-Only` → `ablations` → `ablations_fatigued` → `fatigue` → `cold_start` → `merge`
   (The four non-LLM baselines are scored inside every `eval_` stage; `merge` takes them from the MCGA-LM stage.)
2. From the second session on: **Add Input → Your Work → this notebook's latest output**, so the earlier stages' outputs (including the pre-trained encoder) are carried forward.
3. **Save Version → Save & Run All (Commit).** This runs in the background for up to 12 hours.

**During `pretrain`,** the log reports seconds per epoch and the hours left in each stage. If stage 1 alone would take more than about 5 hours, stop, set `PRETRAIN_MAX_WINDOWS` (e.g. 300) and run it again.

**After `pilot`, stop and check its output.** It prints IHR@3 and an estimate of how long each later stage takes. If retrieval is at chance, the later stages would waste GPU time. If a stage would exceed about 11 hours, split it by seed, e.g. `eval_MCGA-LM@0,1` then `eval_MCGA-LM@2,3,4`. `merge` recombines the parts.

Every number that goes into the paper must come from the final output of this notebook.

In [1]:
STAGE = "pretrain"            # <- set before each session (see the order above)
COMMIT = "61bc265"  # the pushed commit with the loader fixes; keep the same for every stage
CFG = "configs/llm_hf.yaml"
PRETRAIN_MAX_WINDOWS = None   # e.g. 300 if the pretrain log projects more than ~5 h for stage 1

import collections, datetime, glob, json, os, pathlib, shutil, subprocess, sys, time

OUT  = pathlib.Path("/kaggle/working/out")
CODE = pathlib.Path("/tmp/mcga/code")
DATA = pathlib.Path("/tmp/mcga/data")
PY = sys.executable
OUT.mkdir(parents=True, exist_ok=True)
if not COMMIT.startswith("61bc265"):
    raise SystemExit("set COMMIT to the pushed commit that contains the loader fixes")
print("stage:", STAGE)

stage: pretrain


In [2]:
# Carry forward outputs from earlier sessions (attached via Add Input -> Your Work).
found = glob.glob("/kaggle/input/**/out/STAGES_DONE.json", recursive=True)
if found:
    best = max(found, key=lambda f: len(json.loads(pathlib.Path(f).read_text())))
    if len(found) > 1:
        print("several earlier outputs are attached; using the one with the most stages:", best)
    shutil.copytree(pathlib.Path(best).parent, OUT, dirs_exist_ok=True)
    print("restored outputs from", pathlib.Path(best).parent)
done = json.loads((OUT / "STAGES_DONE.json").read_text()) if (OUT / "STAGES_DONE.json").exists() else {}
print("stages already done:", list(done) or "none")
if STAGE in done:
    raise SystemExit(f"stage {STAGE} is already done; pick the next stage")
for name, d in done.items():
    if d["commit"][:7] != COMMIT[:7]:
        raise SystemExit(f"stage {name} used commit {d['commit']}; keep COMMIT the same for every stage")

stages already done: none


In [3]:
def run(cmd, log_name):
    """Run a command from the code checkout, stream and log its output, stop on failure."""
    (OUT / "logs").mkdir(exist_ok=True)
    log = OUT / "logs" / f"{log_name}.log"
    print("$", " ".join(map(str, cmd)))
    t0 = time.time()
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}  # show progress as it happens
    with open(log, "a") as fh:
        p = subprocess.Popen(list(map(str, cmd)), cwd=CODE, env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end=""); fh.write(line); fh.flush()
        p.wait()
    dt = time.time() - t0
    print(f"[{dt/3600:.2f} h]")
    if p.returncode != 0:
        raise SystemExit(f"command failed ({p.returncode}); see {log}")
    return dt

# Code at the pinned commit
if not CODE.exists():
    subprocess.run(["git", "clone", "https://github.com/aasimwadood/MCGA-LM.git", str(CODE)], check=True)
subprocess.run(["git", "fetch", "--quiet", "origin"], cwd=CODE, check=True)
subprocess.run(["git", "checkout", "--quiet", COMMIT], cwd=CODE, check=True)
commit = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
subprocess.run([PY, "-m", "pip", "install", "-q", "-e", f"{CODE}[llm,dev]"], check=True)

import torch
if not torch.cuda.is_available():
    raise SystemExit("no GPU: Settings -> Accelerator -> GPU T4 x2")

# Hugging Face login from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(token=UserSecretsClient().get_secret("HF_TOKEN"))

# Provenance for this stage
prov = OUT / "provenance" / STAGE
prov.mkdir(parents=True, exist_ok=True)
(prov / "commit.txt").write_text(commit + "\n")
(prov / "pip_freeze.txt").write_text(subprocess.run([PY, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
(prov / "nvidia_smi.txt").write_text(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
shutil.copy(CODE / CFG, prov / "config.yaml")
print("commit", commit)

commit 61bc26533acf2b1613767679b1bafc22d86daa8c


In [4]:
CKPT = OUT / "pretrain" / "encoder_pretrained.pt"
RAW = CODE / "data" / "raw"

def fetch_public_data():
    RAW.mkdir(parents=True, exist_ok=True)
    DATA.mkdir(parents=True, exist_ok=True)
    curl = ["curl", "-fL", "--retry", "3", "-o"]   # -f: an HTTP error fails instead of saving an error page
    # WESAD (CC BY 4.0). UCI's download is only a text file pointing to this archive (2.2 GB).
    (RAW / "wesad").mkdir(exist_ok=True)
    if not list((RAW / "wesad").rglob("S*.pkl")):
        z = DATA / "wesad.zip"
        subprocess.run(curl + [z, "https://uni-siegen.sciebo.de/s/HGdUkoNlW1Ub0Gx/download"], check=True)
        subprocess.run(["unzip", "-q", "-o", z, "-d", DATA / "wesad"], check=True); z.unlink()
        for pkl in (DATA / "wesad").rglob("S*.pkl"):
            (RAW / "wesad" / pkl.stem).mkdir(exist_ok=True)
            shutil.move(str(pkl), RAW / "wesad" / pkl.stem / pkl.name)
    # MAMEM Phase I (figshare, CC BY 4.0): Part 1 4.0 GB, Part 2 1.9 GB
    (RAW / "mamem").mkdir(exist_ok=True)
    if not list((RAW / "mamem").rglob("*.mat")):
        for i, url in enumerate(["https://ndownloader.figshare.com/files/8946247",
                                 "https://ndownloader.figshare.com/files/8964538"]):
            z = DATA / f"mamem{i}.zip"
            subprocess.run(curl + [z, url], check=True)
            subprocess.run(["unzip", "-q", "-o", z, "-d", RAW / "mamem"], check=True); z.unlink()
    # CLAS: link the whole CLAS_Database folder from your private dataset
    src = glob.glob("/kaggle/input/**/CLAS_Database", recursive=True)
    if not src:
        raise SystemExit("CLAS_Database not found under /kaggle/input; attach your private CLAS dataset")
    if (RAW / "clas").is_symlink():
        (RAW / "clas").unlink()
    elif (RAW / "clas").exists():
        shutil.rmtree(RAW / "clas")
    os.symlink(src[0], RAW / "clas")

def check_public_data():
    """Fail in minutes, not hours: every corpus present, and CLAS block types readable."""
    n_wesad = len(list((RAW / "wesad").glob("S*/S*.pkl")))
    n_mamem = len([p for p in (RAW / "mamem").rglob("*.mat") if "__MACOSX" not in p.parts])
    print(f"WESAD: {n_wesad} subjects   MAMEM: {n_mamem} session files")
    if n_wesad == 0 or n_mamem == 0:
        raise SystemExit("WESAD or MAMEM is missing; see the download output above")
    sys.path.insert(0, str(CODE / "src"))
    from mcga_lm.data.public_datasets import clas_inventory
    inv = clas_inventory(RAW / "clas")
    print("CLAS:", json.dumps(inv, indent=1))
    for kind in ("ecg", "gsr_ppg"):  # the first lines of one file of each kind, for the record
        f = next((RAW / "clas").rglob(f"*_{kind}_.csv"), None)
        if f:
            print(f"{f.name}:", " | ".join(l.strip() for l in f.open(errors="ignore").readlines()[:2]))
    if inv["blocks"] == 0 or inv["typed_blocks"] < 0.9 * inv["blocks"] or inv["high_load_blocks"] == 0:
        raise SystemExit("CLAS block types were not read as expected; send the 'CLAS:' output above")

if STAGE != "pretrain" and not CKPT.exists():
    raise SystemExit("no pre-trained encoder: run the 'pretrain' stage first and attach its output")

In [ ]:
GENERATIVE = ["MCGA-LM", "RAG-LLM", "M-LLM", "LLM-Only"]
stage, _, seed_part = STAGE.partition("@")
seed_args = ["--set", f"simulation.seeds=[{seed_part}]"] if seed_part else []
chunk = STAGE.replace("@", "_seeds").replace(",", "-")

if stage == "pretrain":
    fetch_public_data()
    check_public_data()
    cap = ["--max-windows-per-recording", str(PRETRAIN_MAX_WINDOWS)] if PRETRAIN_MAX_WINDOWS else []
    secs = run([PY, "scripts/pretrain_encoder.py", "--config", CFG, "--out", OUT,
                "--data-root", "data/raw", "--require-public-data", *cap], STAGE)
    by_source = json.loads((OUT / "pretrain" / "pretrain_report.json").read_text()).get("records_by_source", {})
    print("windows per corpus:", by_source)
    missing = {"mamem", "clas", "wesad"} - set(by_source)
    if missing:
        raise SystemExit(f"pre-training used no windows from {sorted(missing)}")
elif stage == "pilot":
    secs = run([PY, "scripts/evaluate.py", "--config", CFG, "--out", OUT / "pilot", "--pretrained", CKPT,
                "--systems", "MCGA-LM", "--set", "simulation.seeds=[0]", "--skip-heldout", "--descriptive-only"], STAGE)
    r = json.loads((OUT / "pilot" / "evaluate" / "results.json").read_text())["results"]["MCGA-LM"]["aggregate"]
    print("\nPILOT (MCGA-LM, 1 seed):", {k: round(v["mean"], 3) for k, v in r.items() if k in ("ihr@1", "ihr@3", "sact", "wpm", "far")})
    print(f"One system x 1 seed took {secs/3600:.2f} h, so one system x 5 seeds takes up to ~{5*secs/3600:.1f} h.")
    print("If that exceeds ~11 h, split each eval stage by seed, e.g. eval_MCGA-LM@0,1 then eval_MCGA-LM@2,3,4.")
    print("If IHR@3 is near chance, stop here: retrieval is not learning, and the later stages would not be meaningful.")
elif stage.startswith("eval_"):
    system = stage[len("eval_"):]
    if system not in GENERATIVE:
        raise SystemExit(f"unknown system {system!r}: use one of {GENERATIVE}; "
                         "the non-LLM baselines run inside every eval stage")
    secs = run([PY, "scripts/evaluate.py", "--config", CFG, "--out", OUT / "evaluate_chunks" / chunk,
                "--pretrained", CKPT, "--systems", system, *seed_args], STAGE)
elif stage == "ablations":
    secs = run([PY, "scripts/run_ablations.py", "--config", CFG, "--out", OUT, "--pretrained", CKPT], STAGE)
elif stage == "ablations_fatigued":
    secs = run([PY, "scripts/run_ablations.py", "--config", CFG, "--out", OUT / "fatigued", "--pretrained", CKPT, "--fatigued"], STAGE)
elif stage == "fatigue":
    secs = run([PY, "scripts/run_fatigue_study.py", "--config", CFG, "--out", OUT, "--pretrained", CKPT], STAGE)
elif stage == "cold_start":
    secs = run([PY, "scripts/run_cold_start.py", "--config", CFG, "--out", OUT, "--pretrained", CKPT], STAGE)
elif stage == "merge":
    secs = run([PY, "scripts/merge_results.py", "--chunks", OUT / "evaluate_chunks",
                "--config", CFG, "--out", OUT / "evaluate_merged"], STAGE)
else:
    raise SystemExit(f"unknown stage {STAGE}")

WESAD: 15 subjects   MAMEM: 184 session files
CLAS: {
 "participants": 60,
 "blocks": 1920,
 "typed_blocks": 1770,
 "high_load_blocks": 118,
 "block_types": {
  "baseline": 59,
  "iq test response": 59,
  "math test response": 59,
  "neutral": 590,
  "stroop test response": 59,
  "video clip": 944
 },
 "participants_without_block_details": [
  "4"
 ],
 "folders_without_participant": [
  "/tmp/mcga/code/data/raw/clas/CLAS/Participants/Sample/Sample/by_block"
 ],
 "duplicate_files": 64
}
19_ecg_.csv: Timestamp,PythonTimestamp,ecg1,ecg2 | 12175120,1552550833.0944617,-605.0000721216288,5.482325611391737
35_gsr_ppg_.csv: Timestamp,PythonTimestamp,accelx,accely,accelz,ppg,gsr | 11720822,1552551703.296124,2187,2373,1304,1368.4981684981683,166.4917127071823
$ /usr/bin/python3 scripts/pretrain_encoder.py --config configs/llm_hf.yaml --out /kaggle/working/out --data-root data/raw --require-public-data

Stage 1-2: encoder pre-training (paper Sec. 3.7)
[INFO] mcga_lm.data.public_datasets: mamem: 2

In [ ]:
done[STAGE] = {
    "commit": commit,
    "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "hours": round(secs / 3600, 3),
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip(),
}
(OUT / "STAGES_DONE.json").write_text(json.dumps(done, indent=2))
print("\nrecorded stage", STAGE, "->", OUT / "STAGES_DONE.json")